# **LABORATORIO CALIFICADO 1**

----

📋 ***Información General***

**Institución:** TECSUP - Pasión por la Tecnología  
**Curso:** Analítica Empresarial Integrada  
**Docente:** Pilar Rocío Sayán Mejía  
**Semestre:** 2026-II  


👥 ***Integrantes***

  * Alvino Carhuaricra
  * Cordova Agurto, Isabel
  * Palomino Acuña, Luis
  * Vasquez Espinoza, Jesus


---

# **FASE 1 CRISP-DM: Comprensión del Negocio**

## **1. Contexto del Negocio**

## Caso: Andes Supply S.A.C.

Andes Supply S.A.C. es una empresa peruana que provee bienes y servicios industriales a las compañías mineras del país: repuestos, mantenimiento de equipo pesado y servicios de logística en mina. Vende a crédito y actualmente otorga a todos sus clientes las mismas condiciones: noventa días para pagar.

En 2025 dos clientes dejaron facturas impagas y la gerencia general decidió que, a partir de 2026, las condiciones de crédito dejarán de ser iguales para todos. El área comercial debe clasificar a cada cliente minero en uno de tres tramos: crédito a 90 días, crédito a 30 días o pago adelantado. Además, debe sustentar la clasificación ante el cliente.

Andes Supply no cuenta con información interna de sus clientes. Solo puede acceder a la información financiera pública que las mineras presentan ante la Superintendencia del Mercado de Valores.

**Pregunta de negocio:** ¿qué condiciones de crédito debe otorgar Andes Supply a cada cliente minero, con qué evidencia lo sustenta y qué no puede afirmar con la información que tiene?

Los estados financieros de las mineras y las cotizaciones del BCRP son datos reales, oficiales y públicos. Andes Supply es una empresa ficticia creada únicamente para contextualizar la decisión.

## **2. Pregunta del Negocio**

¿Qué condiciones de crédito debe otorgar Andes Supply a  cada cliente minero, con qué evidencia lo sustenta, y qué no puede afirmar con  la información que tiene?


# **FASE 2 CRISP-DM: Comprensión de los datos**

## **Ejercicio 1 — Ficha de trazabilidad del activo de datos (2 puntos)**

**Resultado exigido.** Documente el alcance real del conjunto de datos que acaba de descargar. La gerencia debe poder saber, sin abrir el código, de dónde proviene la información y hasta dónde llega.

**La entrega debe contener:**

- La fuente y el servicio oficial efectivamente consultados, y el ejercicio económico al que corresponden los estados financieros.
- La cantidad de empresas distintas que devuelve la consulta de información financiera.
- La cantidad de sectores económicos distintos presentes en esa respuesta.
- La cantidad de cuentas contables distintas que trae el estado de situación financiera.
- Las monedas en que reportan las empresas del conjunto.

**Criterio de aceptación.** Los cinco valores numéricos deben obtenerse por cálculo sobre las tablas descargadas. Si alguno aparece escrito literalmente en el código, el criterio se considera no logrado.

### **2.1. Librerías**

In [1]:
# ================
#  2.1. LIBRERÍAS
# ================

# Instalación de dependencias específicas para garantizar la reproducibilidad
%pip install -q polars==1.17.1 duckdb==1.1.3

# Importación de librerías estándar y de terceros
import json, html, re
from xml.etree import ElementTree as ET
import requests
import polars as pl          # Motor principal de procesamiento de datos
import pandas as pd          # solo para recibir la respuesta de la SMV
import plotly.express as px  # Visualización de datos
from IPython.display import display

# Configuración de visualización de Polars para mostrar más filas y columnas en el notebook
pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_width_chars(180)

print("polars:", pl.__version__)
print("Entorno listo.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 73.8 MB/s eta 0:00:00
polars: 1.17.1
Entorno listo.


**Interpretación**

En este bloque se prepara el entorno de trabajo para realizar el análisis de manera reproducible, se establecen versiones específicas de Polars y DuckDB para reducir posibles diferencias en el comportamiento del código si las librerías cambian en el futuro. Polars será utilizado como herramienta principal para la manipulación y análisis de los datos, mientras que DuckDB permitirá realizar consultas sobre los conjuntos de datos cuando sea necesario.

También se importan las librerías necesarias para las siguientes etapas del proceso, requests permitirá realizar las solicitudes al servicio de información de la SMV, mientras que ElementTree, json, html y re proporcionan herramientas para procesar y transformar la respuesta obtenida, Pandas se mantiene únicamente para recibir inicialmente la información proveniente de la SMV antes de continuar el procesamiento con Polars.

Finalmente, se configura la visualización de las tablas de Polars para mostrar hasta 25 filas y ampliar el espacio disponible para las columnas, esto facilita la revisión inicial de los datos y permite identificar de forma más sencilla posibles diferencias en su estructura o contenido antes de realizar los cálculos del ejercicio.

El resultado confirma que el entorno se encuentra correctamente configurado y que Polars está trabajando con la versión 1.17.1 establecida para el laboratorio.

### **2.2. Configuración del entorno y parámetros**

In [2]:
# ============================================
# 2.2. CONFIGURACIÓN DEL ENTORNO Y PARÁMETROS
# ============================================

# Centralizamos todos los parámetros variables en un solo lugar.
# Esto garantiza la reproducibilidad: si cambia el año o el sector,
# solo se modifica aquí y todo el notebook se actualiza automáticamente.

# Fuente de datos oficial
SERVICIO_SMV = "https://mvnet.smv.gob.pe/ws_od_eeff/WebServiceInfoFinanciera.asmx"

# Parámetros de la consulta a la SMV (Ejercicio 2024, Periodo Anual, Tipo Individual)
EJERCICIO = 2024
PERIODO = "A"   # Anual
TIPO = "I"      # Individual
SECTOR_OBJETIVO = "MINERAS"

# Codigos oficiales de cuenta del Estado de Situacion Financiera.
# Se identifican por CODIGO y no por descripcion: existen cuentas cuyo texto
# tambien contiene "Activos Corrientes" sin ser el total.
TOTAL_ACTIVO_CORRIENTE = "1D01ST"
TOTAL_PASIVO_CORRIENTE = "1D03ST"

print("Parámetros de configuración definidos correctamente.")

Parámetros de configuración definidos correctamente.


**Interpretación**

En este bloque se centralizan los parámetros que controlan la consulta y el análisis, como el año del ejercicio, el tipo de periodo, la modalidad de los estados financieros y el sector que se desea estudiar, esta decisión permite modificar los criterios de análisis desde un único lugar, facilitando el mantenimiento y la reproducción del procedimiento sin tener que buscar y cambiar los mismos valores en diferentes partes del código.

También se establece directamente la dirección del servicio oficial de información financiera de la SMV, que será la fuente utilizada para obtener los datos del análisis, para este ejercicio se selecciona el año 2024, el periodo anual y los estados financieros individuales del sector de empresas mineras.

Por otro lado, las cuentas de Activo Corriente y Pasivo Corriente se identifican mediante sus códigos contables oficiales (1D01ST y 1D03ST) en lugar de utilizar la descripción de la cuenta, esto permite identificar de manera más precisa las cuentas que se necesitan, ya que una búsqueda basada únicamente en el texto podría encontrar otras cuentas que contengan palabras similares y generar resultados incorrectos.

El resultado confirma que los parámetros necesarios para realizar la consulta quedaron definidos correctamente y podrán ser utilizados por las siguientes etapas del notebook.

### **2.3. Carga de Datos y Revisión Inicial**

In [3]:
# ==============================================================================
# 2.3. CARGA DE DATOS Y REVISIÓN INICIAL (Conexión a la SMV)
# ==============================================================================

print("\n" + "=" * 17)
print(" Revisión inicial")
print("=" * 17, "\n")

def _local(etiqueta):
    """
    Elimina el namespace XML de las etiquetas (ej: '{http://...}Tag' -> 'Tag').
    Es necesario porque la librería ElementTree de Python incluye el namespace
    por defecto, lo que dificultaría buscar las columnas por su nombre simple.
    """
    return str(etiqueta).split("}")[-1]

def _a_dataframe(payload):
    """
    Transforma los datos que nos envía directamente el servicio de la SMV
    en un DataFrame de Pandas. Primero intentamos leer la información como JSON,
    que es el formato más común. Si eso da error, usamos una alternativa para
    leerlo como XML, ya que a veces el servicio entrega los datos con esa
    estructura en lugar de texto simple.
    """
    payload = html.unescape(str(payload)).strip()
    try:
        objeto = json.loads(payload)
        if isinstance(objeto, dict):
            objeto = objeto.get("rows", objeto.get("data", objeto))
        if isinstance(objeto, dict):
            objeto = [objeto]
        return pd.DataFrame(objeto)
    except json.JSONDecodeError:
        raiz = ET.fromstring(payload)
        registros = []
        for nodo in raiz.iter():
            hijos = list(nodo)
            # Filtramos nodos que sean hojas del árbol XML (tienen 5+ hijos y no tienen sub-hijos)
            if len(hijos) >= 5 and all(not list(h) for h in hijos):
                registros.append({_local(h.tag): h.text for h in hijos})
        return pd.DataFrame(registros)

def descargar_smv(operacion, ejercicio, periodo="A", tipo="I"):
    """
    Ejecuta la petición POST SOAP al servicio de la SMV.
    Se usa 'requests' con un timeout de 180s porque los servicios gubernamentales
    pueden tener latencia alta al devolver miles de registros contables.
    """
    sobre = f"""<?xml version="1.0" encoding="utf-8"?>
    <soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
      <soap:Body><{operacion} xmlns="http://tempuri.org/">
        <Ejercicio>{ejercicio}</Ejercicio><Periodo>{periodo}</Periodo><Tipo>{tipo}</Tipo>
      </{operacion}></soap:Body></soap:Envelope>"""

    respuesta = requests.post(
        SERVICIO_SMV,
        data=sobre.encode("utf-8"),
        timeout=180,
        headers={
            "Content-Type": "text/xml; charset=utf-8",
            "SOAPAction": f'"http://tempuri.org/{operacion}"'
        }
    )
    respuesta.raise_for_status() # Lanza error si el servidor devuelve 4xx o 5xx

    raiz = ET.fromstring(respuesta.content)
    resultado = next((n for n in raiz.iter() if _local(n.tag) == f"{operacion}Result"), None)

    if resultado is None or not (resultado.text or "").strip():
        raise ValueError(f"La SMV no devolvió datos para la operación: {operacion}.")

    # Convertimos a Polars inmediatamente para unificar el stack de procesamiento
    return pl.from_pandas(_a_dataframe(resultado.text))

# Ejecución de las descargas oficiales
principales = descargar_smv("obtener_InfoFinanciera", EJERCICIO, PERIODO, TIPO)
balance = descargar_smv("obtener_BalanceGeneral", EJERCICIO, PERIODO, TIPO)

print(f"🔹 Cuentas principales descargadas: {principales.shape}")
print(f"🔹 Estado de situación financiera descargado: {balance.shape} \n")

display(principales.head(3))


 Revisión inicial

🔹 Cuentas principales descargadas: (276, 16)
🔹 Estado de situación financiera descargado: (20574, 15) 



RPJ,TipoEmpresa,TipoSector,NombreEmpresa,RUC,CIIU,Ejercicio,TipoInformacion,Trimestre,Moneda,MetodoFlujoEfectivo,ActivoTotal,PatrimonioTotal,TotalIngreso,UtilidadNeta,PasivoTotal
str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64
"""L00474""","""EMPRESAS MERCADO ALTERNATIVO D…","""DIVERSOS""","""A. JAIME ROJAS REPRESENTACIONE…","""20102032951""","""5190""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Directo""",77710,39497,77196,6786,38213
"""I00004""","""SOCIEDADES ADMINISTRADORAS DE …","""""","""AC CAPITALES SOCIEDAD ADMINIST…","""20504893295""","""6430""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Directo""",9070,7373,4880,-85,1697
"""OE7511""","""SOCIEDADES ADMINISTRADORAS DE …","""""","""ACRES SOCIEDAD ADMINISTRADORA …","""20601498996""","""6430""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Indirecto""",2724,2238,2348,176,486


**Interpretación**

En este bloque se establece la conexión con el servicio oficial de información financiera de la SMV y se construyen funciones para descargar y transformar la información obtenida, debido a que el servicio utiliza el protocolo SOAP, se genera una solicitud XML y se envía mediante requests; además, se establece un tiempo máximo de espera de 180 segundos, considerando que la consulta puede devolver una cantidad importante de registros.

La función _a_dataframe() permite transformar la respuesta recibida en una estructura tabular, primero intenta interpretar el contenido como JSON y, si no corresponde a ese formato, utiliza XML como alternativa, esta conversión permite continuar posteriormente el procesamiento utilizando Polars, también se eliminan los espacios y los namespaces de las etiquetas XML para facilitar la identificación de los campos.

Como resultado, se obtuvieron dos tablas provenientes del servicio de la SMV, la tabla de información financiera contiene 276 registros y 16 columnas, mientras que el Estado de Situación Financiera contiene 20,574 registros y 15 columnas. La vista previa permite comprobar que la información contiene datos financieros como Activo Total, Patrimonio Total, Ingresos, Utilidad Neta y Pasivo Total, además de información de identificación de las empresas, ejercicio, tipo de información y moneda.

En esta etapa todavía no se considera que las 276 filas correspondan necesariamente a 276 empresas distintas, ya que la cantidad de empresas debe determinarse utilizando un identificador de empresa y realizando el cálculo correspondiente sobre los datos descargados; asimismo, se observa que la consulta inicial contiene empresas de diferentes sectores, por lo que será necesario aplicar posteriormente el filtro correspondiente al sector minero antes de construir el perfil comparativo solicitado.

### **2.4. Ficha de trazabilidad del activo de datos**

In [4]:
# ============================================================
# 2.4. Ejercicio 1: FICHA DE TRAZABILIDAD DEL ACTIVO DE DATOS
# ============================================================

print("\n" + "=" * 43)
print(" Ficha de trazabilidad del activo de datos")
print("=" * 43, "\n")

# Esta ficha documenta el alcance real del conjunto de datos descargado.
# Los cinco valores numéricos se calculan dinámicamente sobre las tablas
# descargadas. Ningún valor está escrito a mano, lo que
# garantiza la reproducibilidad, si la SMV cambia la cantidad de empresas
# o cuentas, el código se actualiza automáticamente al reejecutar.

FICHA = {
    "fuente": "SMV - Portal de Datos Abiertos",
    "servicio": SERVICIO_SMV,
    "ejercicio": EJERCICIO,

    # Contamos empresas únicas (no filas totales) porque una empresa puede
    # aparecer varias veces si reporta en diferentes periodos o monedas.
    "empresas_totales": principales["NombreEmpresa"].n_unique(),

    # Contamos sectores únicos para entender la diversidad del dataset original.
    "sectores_distintos": principales["TipoSector"].n_unique(),

    # Contamos códigos de cuenta únicos en el balance general.
    "cuentas_balance": balance["Cuenta"].n_unique(),

    # Extraemos las monedas únicas y las ordenamos alfabéticamente para legibilidad.
    "monedas": sorted(principales["Moneda"].unique().to_list()),
}

# Imprimimos la ficha completa con formato alineado para facilitar la lectura.
for k, v in FICHA.items():
    print(f"{k:>20}: {v}")

# Advertencia metodológica: mostramos la diferencia entre contar filas
# (principales.height) vs. contar empresas únicas (n_unique).
# Esto demuestra que entendemos la estructura de los datos.
print("\n" + "=" * 24)
print(" Validación de métricas")
print("=" * 24, "\n")

print(f"Filas totales de 'principales': {principales.height}")
print(f"Empresas distintas por RUC: {principales['RUC'].n_unique()}")
print(f"Empresas distintas por nombre: {principales['NombreEmpresa'].n_unique()}")


 Ficha de trazabilidad del activo de datos

              fuente: SMV - Portal de Datos Abiertos
            servicio: https://mvnet.smv.gob.pe/ws_od_eeff/WebServiceInfoFinanciera.asmx
           ejercicio: 2024
    empresas_totales: 276
  sectores_distintos: 10
     cuentas_balance: 482
             monedas: ['D lares', 'Soles']

 Validación de métricas

Filas totales de 'principales': 276
Empresas distintas por RUC: 274
Empresas distintas por nombre: 276


In [5]:
print("\n" + "=" * 17)
print(" Análisis de RUC")
print("=" * 17, "\n")

# Identificamos los RUC que aparecen en más de un registro.
duplicados_ruc = (
    principales
    .group_by("RUC")
    .agg(
        pl.len().alias("cantidad_registros"),
        pl.col("NombreEmpresa").n_unique().alias("nombres_asociados")
    )
    .filter(pl.col("cantidad_registros") > 1)
)

display(duplicados_ruc)




 Análisis de RUC



RUC,cantidad_registros,nombres_asociados
str,u32,u32
"""0""",3,3


In [6]:
print("\n" + "=" * 37)
print(" Muestreo de registros RUC repetidos")
print("=" * 37, "\n")

# Mostramos los registros asociados a RUC repetidos
ruc_repetidos = duplicados_ruc["RUC"].to_list()

display(
    principales
    .filter(pl.col("RUC").is_in(ruc_repetidos))
    .sort("RUC")
)


 Muestreo de registros RUC repetidos



RPJ,TipoEmpresa,TipoSector,NombreEmpresa,RUC,CIIU,Ejercicio,TipoInformacion,Trimestre,Moneda,MetodoFlujoEfectivo,ActivoTotal,PatrimonioTotal,TotalIngreso,UtilidadNeta,PasivoTotal
str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64
"""B60051""","""EMPRESAS EMISORAS""","""DIVERSOS""","""CREDICORP LTD.""","""0""","""6599""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Directo""",40655832,38595515,93486,6185851,2060317
"""OE5087""","""EMPRESAS EMISORAS""","""DIVERSOS""","""INRETAIL PERU CORP.""","""0""","""6719""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Directo""",8282077,5037093,499967,428118,3244984
"""OE2305""","""EMPRESAS EMISORAS""","""DIVERSOS""","""INTERCORP FINANCIAL SERVICES I…","""0""","""6599""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Indirecto""",12043873,10915235,8946,1300078,1128638


In [7]:
print("\n" + "=" * 24)
print(" Validación de métricas")
print("=" * 24, "\n")

print(f"Filas totales de 'principales': {principales.height}")

# Contamos los RUC válidos, excluyendo "0", que no permite identificar
# individualmente a una empresa.
empresas_ruc_validos = (
    principales
    .filter(pl.col("RUC") != "0")
    ["RUC"]
    .n_unique()
)

# Contamos los registros que presentan un RUC no válido.
registros_ruc_no_validos = (
    principales
    .filter(pl.col("RUC") == "0")
    .height
)

print(f"Empresas con RUC válido: {empresas_ruc_validos}")
print(f"Registros con RUC no válido: {registros_ruc_no_validos}")
print(f"Empresas distintas por nombre: {principales['NombreEmpresa'].n_unique()}")


 Validación de métricas

Filas totales de 'principales': 276
Empresas con RUC válido: 273
Registros con RUC no válido: 3
Empresas distintas por nombre: 276


In [8]:
print("\n" + "=" * 31)
print(" Validación de identificadores")
print("=" * 31, "\n")

print(f"Registros totales: {principales.height}")
print(f"RUC distintos: {principales['RUC'].n_unique()}")
print(f"RUC válidos distintos: {principales.filter(pl.col('RUC') != '0')['RUC'].n_unique()}")
print(f"RPJ distintos: {principales['RPJ'].n_unique()}")
print(f"Nombres distintos: {principales['NombreEmpresa'].n_unique()}")


 Validación de identificadores

Registros totales: 276
RUC distintos: 274
RUC válidos distintos: 273
RPJ distintos: 276
Nombres distintos: 276


In [9]:
display(
    principales
    .filter(pl.col("RUC") == "0")
    .select(["RPJ", "NombreEmpresa", "RUC", "TipoSector", "Ejercicio", "Moneda"])
)

RPJ,NombreEmpresa,RUC,TipoSector,Ejercicio,Moneda
str,str,str,str,str,str
"""B60051""","""CREDICORP LTD.""","""0""","""DIVERSOS""","""2024""","""Soles"""
"""OE5087""","""INRETAIL PERU CORP.""","""0""","""DIVERSOS""","""2024""","""Soles"""
"""OE2305""","""INTERCORP FINANCIAL SERVICES I…","""0""","""DIVERSOS""","""2024""","""Soles"""


**Interpretación**

La ficha de trazabilidad permite documentar el origen y el alcance real del conjunto de datos utilizado, la información fue obtenida del servicio oficial de información financiera de la SMV y corresponde al ejercicio económico 2024. La consulta contiene 276 registros y 276 identificadores RPJ distintos, por lo que se identifican 276 entidades diferentes en la tabla de información financiera; además, se encuentran 10 sectores económicos distintos, 482 códigos de cuenta contable en el Estado de Situación Financiera y dos monedas de reporte: dólares y soles.

Durante la validación de los identificadores se encontró que tres registros presentan el valor "0" en la columna RUC; sin embargo, estos registros cuentan con RPJ diferentes y corresponden a empresas diferentes, por lo que el RUC no resulta adecuado como único identificador para determinar la cantidad de entidades del conjunto. En este caso, el RPJ permite distinguir correctamente los 276 registros.

La presencia de dos monedas de reporte también constituye una consideración importante para el análisis posterior, los montos financieros expresados en dólares y soles no deben compararse o agregarse directamente entre empresas sin realizar previamente una conversión a una moneda común. Por este motivo, cuando se trabajen importes absolutos entre empresas será necesario considerar esta diferencia, mientras que los ratios financieros pueden facilitar la comparación cuando sus componentes correspondan al mismo período y estén expresados bajo condiciones equivalentes.

Finalmente, los valores de la ficha se obtienen mediante cálculos realizados directamente sobre las tablas descargadas y no mediante números ingresados manualmente. Esto permite reproducir el procedimiento y recalcular las métricas al volver a ejecutar la consulta. La ficha establece el alcance del activo de datos, pero esta información financiera pública no permite determinar por sí sola si una empresa cumplirá o incumplirá futuras obligaciones de pago; únicamente proporciona evidencia para evaluar determinados aspectos de su situación financiera.


# **FASE 3 CRISP-DM: Preparación de Datos**

## **Ejercicio 2 — Delimitación del sector evaluable (2 puntos)**

**Resultado exigido.** Obtenga la tabla de empresas mineras sobre la que se construirá todo el análisis posterior. No todas las empresas que devuelve el servicio son analizables: algunas pertenecen a otros sectores, otras reportan periodos que no son anuales y otras presentan cuentas incompletas.

**La entrega debe contener:**

- Solo empresas del sector minero.
- Solo información de periodicidad anual.
- Las cinco magnitudes del análisis — activo total, patrimonio, ingresos, utilidad neta y pasivo total — expresadas como valores numéricos.
- Sin empresas que presenten alguna de esas cinco magnitudes vacía.
- Sin empresas con ingresos iguales a cero, porque impiden calcular el margen.

**Criterio de aceptación.** El notebook debe informar cuántas empresas quedaron disponibles tras la delimitación. Una tabla que conserve el total de empresas descargadas indica que los criterios no se aplicaron.

### **3.1. Delimitación del sector evaluable**

In [10]:
# ======================================
# 3.1 DELIMITACIÓN DEL SECTOR EVALUABLE
# ======================================

print("\n" + "=" * 35)
print(" Delimitación del sector evaluable")
print("=" * 35, "\n")

# Se filtra la data cruda para quedarnos únicamente con empresas mineras
# que tengan reportes anuales completos y válidos para el cálculo de ratios.

# 1. Filtrar exclusivamente el sector minero.
# El caso de negocio de Andes Supply atiende solo a mineras.
# Incluir otros sectores como retail o finanzas sesgaría el análisis, ya que
# tienen estructuras de capital y ciclos operativos completamente distintos.
df_mineras = principales.filter(pl.col("TipoSector") == "MINERAS")

# 2. Filtrar solo reportes de periodicidad anual.
# Los reportes anuales son los que están auditados y consolidados,
# lo que garantiza la calidad, estandarización y comparabilidad de los datos.
df_mineras = df_mineras.filter(pl.col("TipoInformacion").str.contains("Anual"))

# 3. Definir las 5 magnitudes clave y convertirlas a formato numérico (Float64).
# La SMV entrega estos datos como texto. Para poder hacer operaciones
# matemáticas como dividir utilidad entre ingresos, es obligatorio convertirlos a números.
# Usamos strict=False para que, si hay algún texto extraño o celda vacía, se convierta
# en nulo (null) en lugar de detener la ejecución con un error.
magnitudes_clave = ["ActivoTotal", "PatrimonioTotal", "TotalIngreso", "UtilidadNeta", "PasivoTotal"]
for col in magnitudes_clave:
    df_mineras = df_mineras.with_columns(pl.col(col).cast(pl.Float64, strict=False))

# 4. Eliminar filas que tengan datos faltantes (nulos) en estas 5 columnas.
# Una empresa sin Activo o sin Ingresos no puede ser evaluada financieramente.
# Mantenerlas generaría errores de división por cero o resultados nulos en los indicadores.
df_mineras = df_mineras.drop_nulls(subset=magnitudes_clave)

# 5. Descartar empresas con Ingresos iguales a cero.
# la Utilidad Neta entre los Ingresos. Si los ingresos son cero, la operación matemática
# es indefinida (infinito o error), lo cual invalidaría el perfil comparado del sector.
df_mineras = df_mineras.filter(pl.col("TotalIngreso") != 0)

# 6. Validación y reporte del criterio de aceptación.
# Mostramos cuántas empresas quedaron y una vista previa de los datos limpios.

print(f"Empresas mineras analizables tras la delimitación: {df_mineras['RPJ'].n_unique()}")
print("\nVista previa de la tabla limpia (primeras 5 filas):")
display(df_mineras.select(["NombreEmpresa", "Moneda"] + magnitudes_clave).head(5))


 Delimitación del sector evaluable

Empresas mineras analizables tras la delimitación: 14

Vista previa de la tabla limpia (primeras 5 filas):


NombreEmpresa,Moneda,ActivoTotal,PatrimonioTotal,TotalIngreso,UtilidadNeta,PasivoTotal
str,str,f64,f64,f64,f64,f64
"""COMPAÑIA DE MINAS BUENAVENTURA…","""D lares""",4.42137e6,3.390689e6,665978.0,402689.0,1.030681e6
"""COMPAÑIA MINERA PODEROSA S.A.A…","""Soles""",2.542791e6,1.883522e6,2.624541e6,415128.0,659269.0
"""COMPAÑIA MINERA SANTA LUISA S.…","""Soles""",523486.0,344321.0,467521.0,102749.0,179165.0
"""MINERA ANDINA DE EXPLORACIONES…","""Soles""",14050.0,6743.0,5370.0,-181.0,7307.0
"""MINSUR S.A.""","""D lares""",2.499941e6,1.625737e6,1.038312e6,463546.0,874204.0


In [11]:
print("\n" + "=" * 24)
print(" Validación de empresas")
print("=" * 24, "\n")

print(f"Registros disponibles: {df_mineras.height}")
print(f"Empresas distintas por RPJ: {df_mineras['RPJ'].n_unique()}")
print(f"Empresas distintas por nombre: {df_mineras['NombreEmpresa'].n_unique()}")


 Validación de empresas

Registros disponibles: 14
Empresas distintas por RPJ: 14
Empresas distintas por nombre: 14


**Interpretación**

En esta etapa se delimitó el conjunto de datos que será utilizado para el análisis financiero de Andes Supply, primero partimos de la información obtenida de la SMV y aplicamos filtros para conservar únicamente las empresas pertenecientes al sector MINERAS y cuyos reportes correspondan a una periodicidad anual, esta delimitación permite trabajar con información correspondiente al mismo período de referencia y evita mezclar empresas de otros sectores o reportes con diferente periodicidad.

Posteriormente, las cinco magnitudes requeridas para el análisis como Activo Total, Patrimonio Total, Ingresos, Utilidad Neta y Pasivo Total, estas fueron convertidas a formato numérico, también se utilizó strict=False para que los valores que no pudieran convertirse correctamente fueran tratados como nulos y posteriormente identificados mediante el proceso de limpieza; por otro lado, se eliminaron los registros que presentaban algún valor faltante en estas variables, ya que no contar con alguno de estos componentes impediría calcular correctamente los indicadores financieros posteriores.

Finalmente, se excluyeron las empresas cuyos ingresos fueran iguales a cero, debido a que este valor no permite calcular el margen neto, uno de los indicadores que se utilizará en el análisis comparado, después de aplicar todos los criterios, el conjunto se redujo a 14 empresas mineras analizables, frente a las 276 entidades disponibles inicialmente.

La tabla resultante muestra que las cinco magnitudes financieras se encuentran en formato numérico y que no presentan valores vacíos dentro de las variables seleccionadas, también se observa que permanecen registros reportados tanto en soles como en dólares, esta diferencia deberá considerarse al comparar importes absolutos entre empresas, mientras que para los ratios financieros los componentes pueden compararse dentro de cada empresa al encontrarse expresados bajo la misma moneda y período de reporte.

En consecuencia, el conjunto obtenido constituye la base sobre la cual se calcularán los indicadores financieros y se construirá posteriormente el perfil comparado del sector minero.

# **FASE 4 CRISP-DM: Modelado**

## **Ejercicio 3 — Perfil financiero comparado del sector (2 puntos)**

**Resultado exigido.** Construya el perfil que permitirá comparar a los clientes entre sí. Se exigen seis indicadores por empresa: margen neto, rentabilidad sobre activos, rentabilidad sobre patrimonio, razón corriente, razón de endeudamiento y apalancamiento.

**La entrega debe contener:**

- El activo corriente y el pasivo corriente deben extraerse del estado de situación financiera identificando la cuenta por su código oficial y no por su descripción textual.
- Los seis indicadores deben quedar incorporados como columnas del perfil, no impresos sueltos.
- El resultado debe presentarse ordenado y legible, con el nombre de la empresa y su moneda.

**Criterio de aceptación.** Ninguna razón puede resultar igual a cero ni infinita. Si la razón corriente sale cero, la cuenta seleccionada no es la correcta: existen cuentas cuya descripción contiene el texto «Activos Corrientes» sin ser el total, y valen cero.

In [12]:
# ====================================================
# EJERCICIO 3: PERFIL FINANCIERO COMPARADO DEL SECTOR
# ====================================================

print("\n" + "=" * 50)
print(" Perfil financiero comparado con el sector minero")
print("=" * 50, "\n")

# Objetivo:
# Construir un perfil financiero con seis indicadores por empresa,
# utilizando los códigos oficiales de la SMV para identificar el
# Activo Corriente y el Pasivo Corriente.


# 1. EXTRAER EL ACTIVO CORRIENTE
# Se utiliza el código oficial "1D01ST" para identificar el total del
# Activo Corriente. Se evita buscar por descripción textual porque existen
# otras cuentas cuyos nombres contienen "Activos Corrientes" pero no
# representan el total de esta cuenta.

activo_corriente = (
    balance
    .filter(pl.col("Cuenta") == TOTAL_ACTIVO_CORRIENTE)
    .select([
        pl.col("RPJ"),
        pl.col("Monto1")
        .cast(pl.Float64, strict=False)
        .alias("ActivoCorriente")
    ])
)

# 2. EXTRAER EL PASIVO CORRIENTE
# Se utiliza el código oficial "1D03ST" para obtener el total del Pasivo
# Corriente, manteniendo el mismo criterio utilizado para el Activo Corriente.

pasivo_corriente = (
    balance
    .filter(pl.col("Cuenta") == TOTAL_PASIVO_CORRIENTE)
    .select([
        pl.col("RPJ"),
        pl.col("Monto1")
        .cast(pl.Float64, strict=False)
        .alias("PasivoCorriente")
    ])
)

# 3. INTEGRAR LOS VALORES CORRIENTES A LA TABLA DE MINERAS
# Se utiliza RPJ como identificador de la empresa para realizar la unión.
# Esto es más robusto que utilizar el nombre, ya que el nombre de una empresa
# puede presentar variaciones de escritura y no constituye un identificador
# único formal.

perfil = (
    df_mineras
    .join(activo_corriente, on="RPJ", how="left")
    .join(pasivo_corriente, on="RPJ", how="left")
)

# 4. CALCULAR LOS SEIS INDICADORES FINANCIEROS
perfil_financiero = perfil.with_columns([

    # Margen Neto:
    # Mide qué proporción de los ingresos termina como utilidad neta.
    (pl.col("UtilidadNeta") / pl.col("TotalIngreso"))
    .alias("Margen_Neto"),

    # ROA (Return on Assets):
    # Mide la rentabilidad generada en relación con el total de activos.
    (pl.col("UtilidadNeta") / pl.col("ActivoTotal"))
    .alias("ROA"),

    # ROE (Return on Equity):
    # Mide la rentabilidad obtenida en relación con el patrimonio.
    (pl.col("UtilidadNeta") / pl.col("PatrimonioTotal"))
    .alias("ROE"),

    # Razón Corriente:
    # Mide la capacidad de cubrir obligaciones de corto plazo mediante
    # los activos corrientes.
    # Se evita la división entre cero mediante when/otherwise.
    pl.when(pl.col("PasivoCorriente") != 0)
      .then(pl.col("ActivoCorriente") / pl.col("PasivoCorriente"))
      .otherwise(None)
      .alias("Razon_Corriente"),

    # Endeudamiento:
    # Representa la proporción de los activos financiada mediante pasivos.
    (pl.col("PasivoTotal") / pl.col("ActivoTotal"))
    .alias("Endeudamiento"),

    # Apalancamiento:
    # Representa la cantidad de pasivos existente por cada unidad monetaria
    # de patrimonio.
    (pl.col("PasivoTotal") / pl.col("PatrimonioTotal"))
    .alias("Apalancamiento")
])

# 5. SELECCIONAR Y ORDENAR EL PERFIL FINAL
# Se conservan únicamente el nombre, la moneda y los seis indicadores
# requeridos para presentar un perfil compacto y fácil de interpretar.

columnas_texto = [
    "NombreEmpresa",
    "Moneda"
]

columnas_numericas = [
    "Margen_Neto",
    "ROA",
    "ROE",
    "Razon_Corriente",
    "Endeudamiento",
    "Apalancamiento"
]

resultado_ejercicio_3 = (
    perfil_financiero
    .select(
        pl.col(columnas_texto),
        pl.col(columnas_numericas).round(4)
    )
    .sort("NombreEmpresa")
)

# 6. PRESENTAR RESULTADOS
print(
    f"Empresas procesadas exitosamente: "
    f"{resultado_ejercicio_3.height}"
)

display(resultado_ejercicio_3)

# 7. VALIDACIÓN DE LOS SEIS INDICADORES
# Por ninguna razón debe ser igual a cero o infinita.
# Por ello, se revisan los seis indicadores y no únicamente la Razón Corriente.
# También se comprueba que no existan valores nulos.

hay_ceros = (
    resultado_ejercicio_3
    .select([
        pl.col(c) == 0
        for c in columnas_numericas
    ])
    .to_numpy()
    .any()
)

hay_infinitos = (
    resultado_ejercicio_3
    .select([
        pl.col(c).is_infinite()
        for c in columnas_numericas
    ])
    .to_numpy()
    .any()
)

hay_nulos = (
    resultado_ejercicio_3
    .select([
        pl.col(c).is_null()
        for c in columnas_numericas
    ])
    .to_numpy()
    .any()
)

if hay_ceros or hay_infinitos or hay_nulos:

    print(
        "⚠️ ADVERTENCIA: Se detectaron ceros, valores infinitos "
        "o valores nulos en uno o más indicadores."
    )

else:

    print(
        "✅ Validación exitosa: Los seis indicadores presentan "
        "valores numéricos, finitos y distintos de cero."
    )


 Perfil financiero comparado con el sector minero

Empresas procesadas exitosamente: 14


NombreEmpresa,Moneda,Margen_Neto,ROA,ROE,Razon_Corriente,Endeudamiento,Apalancamiento
str,str,f64,f64,f64,f64,f64,f64
"""COMPAÑIA DE MINAS BUENAVENTURA…","""D lares""",0.6047,0.0911,0.1188,1.5211,0.2331,0.304
"""COMPAÑIA MINERA PODEROSA S.A.A…","""Soles""",0.1582,0.1633,0.2204,0.8132,0.2593,0.35
"""COMPAÑIA MINERA SANTA LUISA S.…","""Soles""",0.2198,0.1963,0.2984,3.6666,0.3423,0.5203
"""MINERA ANDINA DE EXPLORACIONES…","""Soles""",-0.0337,-0.0129,-0.0268,2.1267,0.5201,1.0836
"""MINSUR S.A.""","""D lares""",0.4464,0.1854,0.2851,1.4813,0.3497,0.5377
"""NEXA RESOURCES ATACOCHA S.A.A.""","""D lares""",0.1085,0.094,1.1979,1.3316,0.9215,11.7455
"""NEXA RESOURCES PERU S.A.A.""","""D lares""",-0.0281,-0.0146,-0.0207,2.5857,0.2968,0.4221
"""PERUBAR S.A.""","""D lares""",0.0962,0.0258,0.034,0.741,0.2407,0.317
"""SHOUGANG HIERRO PERU S.A.A.""","""Soles""",0.3563,0.1976,0.5508,2.585,0.6413,1.7882


✅ Validación exitosa: Los seis indicadores presentan valores numéricos, finitos y distintos de cero.


**Interpretación**

En este ejercicio se construyó el perfil financiero comparado de las 14 empresas mineras que quedaron disponibles después de la delimitación de los datos, para calcular la Razón Corriente se utilizaron los códigos oficiales de la SMV 1D01ST para el Activo Corriente y 1D03ST para el Pasivo Corriente, en lugar de realizar una búsqueda por descripción, esto permite trabajar con las cuentas totales correspondientes y evitar seleccionar registros que tengan una descripción similar, pero que no representen el valor total de la cuenta.

A partir de la información financiera se calcularon seis indicadores, el Margen Neto, ROA, ROE, Razón Corriente, Endeudamiento y Apalancamiento, los resultados muestran que existe una diferencia importante en el comportamiento financiero de las empresas; por ejemplo, Compañía de Minas Buenaventura presenta un Margen Neto de 60,47 %, mientras que Minsur alcanza 44,64 %; por otro lado, Minera Andina de Exploraciones, Nexa Resources Perú y Sociedad Minera Corona presentan márgenes negativos de -3,37 %, -2,81 % y -3,68 %, respectivamente, lo que refleja que durante el período analizado registraron pérdidas netas.

También se observan diferencias importantes en la rentabilidad sobre los activos y el patrimonio, un caso particular es Nexa Resources Atacocha, que presenta un ROE de 119,79 %, muy superior al resto de empresas analizadas; sin embargo, este resultado debe interpretarse con cautela, debido a que la empresa también presenta un nivel de endeudamiento de 92,15 % y un apalancamiento de 11,75; por ello, un ROE elevado no puede considerarse por sí solo como evidencia de una mayor solidez financiera, ya que puede estar relacionado con una estructura de capital altamente apalancada.

En relación con la liquidez, también se encuentran diferencias relevantes como la Compañía Minera Santa Luisa que presenta una Razón Corriente de 3,67 y Sociedad Minera Cerro Verde de 3,56, mientras que Compañía Minera Poderosa y Perubar presentan valores inferiores a 1, de 0,81 y 0,74, respectivamente, esto indica que la capacidad relativa para cubrir obligaciones de corto plazo no es igual entre las empresas y constituye un elemento que deberá considerarse junto con los demás indicadores al momento de evaluar las condiciones de crédito.

Finalmente, la validación confirma que los seis indicadores calculados presentan valores numéricos, finitos y diferentes de cero, por lo que el perfil financiero cumple con el criterio de aceptación establecido para este ejercicio, este perfil permitirá realizar el ranking financiero y analizar con mayor detalle qué empresas presentan mejores condiciones y cuáles requieren una evaluación más conservadora antes de definir sus condiciones de crédito.


## **Ejercicio 4 — Lectura crítica del ranking (2 puntos)**

**Resultado exigido.** Identifique a la empresa que encabeza el sector por rentabilidad sobre patrimonio y extraiga, para esa misma empresa, su endeudamiento, su apalancamiento, su patrimonio y su pasivo total.

**La entrega debe contener:**

- La empresa debe quedar determinada por el propio cálculo sobre el perfil.
- Las cuatro magnitudes deben corresponder a la empresa identificada, obtenidas de la misma tabla.

**Criterio de aceptación.** El nombre de la empresa no puede aparecer escrito en el código. Si el sector cambiara de composición, la respuesta debería actualizarse sola al reejecutar.

In [13]:
# =========================================
# EJERCICIO 4: LECTURA CRÍTICA DEL RANKING
# =========================================

print("\n" + "=" * 30)
print(" Análisis crítico del ranking")
print("=" * 30, "\n")

# Identificar dinámicamente a la empresa con mayor ROE y extraer sus
# métricas de riesgo para evaluar si su rentabilidad es sostenible o riesgosa.

# 1. Identificamos dinámicamente a la empresa con el mayor ROE.
# Justificación: No escribimos el nombre de la empresa a mano (hardcode).
# Usamos .sort() y .head(1) para que, si la composición del sector cambia
# el próximo año, el código encuentre automáticamente a la nueva líder.
top_roe_df = resultado_ejercicio_3.sort("ROE", descending=True).head(1)
nombre_empresa_top = top_roe_df["NombreEmpresa"].item()

# 2. Extraemos las 4 magnitudes exigidas para esa empresa específica.
# Usamos 'perfil_financiero' (no 'perfil') porque ahí están los indicadores calculados.
# 'perfil' solo tiene las magnitudes originales, pero NO tiene ROE, Endeudamiento, etc.
datos_criticos = perfil_financiero.filter(pl.col("NombreEmpresa") == nombre_empresa_top).select([
    pl.col("NombreEmpresa").alias("Empresa_Analizada"),
    pl.col("ROE").round(4).alias("ROE"),
    pl.col("Endeudamiento").round(4).alias("Endeudamiento"),
    pl.col("Apalancamiento").round(2).alias("Apalancamiento"),
    pl.col("PatrimonioTotal").alias("Patrimonio_Total"),
    pl.col("PasivoTotal").alias("Pasivo_Total")
])

print(f"Empresa identificada dinámicamente con mayor ROE: {nombre_empresa_top}\n")
display(datos_criticos)

# 3. LECTURA GERENCIAL DEL RESULTADO
# Se relaciona el ROE obtenido con el nivel de apalancamiento de la empresa.
# Un ROE elevado no debe interpretarse de forma aislada, especialmente cuando
# está acompañado por un nivel elevado de deuda respecto al patrimonio.

roe_val = datos_criticos["ROE"].item()
apalancamiento_val = datos_criticos["Apalancamiento"].item()

print("\n" + "=" * 19)
print(" Lectura gerencial")
print("=" * 19, "\n")

print(
    f"Un ROE de {roe_val:.1%} es elevado, pero debe interpretarse "
    f"junto con un apalancamiento de {apalancamiento_val:.2f}x."
)
print(
    "Esto indica una elevada participación del financiamiento mediante "
    "pasivos respecto al patrimonio, por lo que el ROE no debe utilizarse "
    "de manera aislada como evidencia de solidez financiera."
)


 Análisis crítico del ranking

Empresa identificada dinámicamente con mayor ROE: NEXA RESOURCES ATACOCHA S.A.A.



Empresa_Analizada,ROE,Endeudamiento,Apalancamiento,Patrimonio_Total,Pasivo_Total
str,f64,f64,f64,f64,f64
"""NEXA RESOURCES ATACOCHA S.A.A.""",1.1979,0.9215,11.75,8459.0,99355.0



 Lectura gerencial

Un ROE de 119.8% es elevado, pero debe interpretarse junto con un apalancamiento de 11.75x.
Esto indica una elevada participación del financiamiento mediante pasivos respecto al patrimonio, por lo que el ROE no debe utilizarse de manera aislada como evidencia de solidez financiera.


**Interpretación**

A partir del perfil financiero construido en el ejercicio anterior, se identificó de manera dinámica a la empresa que presenta el mayor ROE, sin definir previamente su nombre en el código, el resultado corresponde a NEXA RESOURCES ATACOCHA S.A.A., con un ROE de 119,79 %.

A primera vista, este valor podría interpretarse como una rentabilidad excepcional para los accionistas; sin embargo, al analizar conjuntamente los demás indicadores de la misma empresa, se observa que presenta un endeudamiento de 92,15 % y un apalancamiento de 11,75 veces; además, su patrimonio total es de 8 459, mientras que su pasivo total alcanza 99 355, esta diferencia muestra que existe una participación muy elevada de los pasivos en relación con el patrimonio.

Por ello, el ROE más alto del sector no debe interpretarse automáticamente como evidencia de que NEXA RESOURCES ATACOCHA S.A.A. sea la empresa financieramente más sólida, el nivel elevado de apalancamiento constituye un factor que debe considerarse al evaluar su capacidad financiera, ya que una rentabilidad alta sobre un patrimonio relativamente reducido puede estar acompañada de una mayor exposición al financiamiento mediante deuda.

Este resultado es importante para la decisión de crédito de Andes Supply, porque demuestra que un único indicador puede llevar a una conclusión equivocada si se analiza de manera aislada, para establecer las condiciones de crédito será necesario considerar conjuntamente la rentabilidad, el endeudamiento, el apalancamiento y la liquidez de cada empresa.


# **FASE 5 CRISP-DM: Evaluación**

## **Ejercicio 5 — Política de crédito de Andes Supply (2 puntos)**

**Resultado exigido.** Clasifique a cada cliente minero en uno de tres tramos de condiciones comerciales: crédito a noventa días, crédito a treinta días o pago adelantado. Esta es la decisión que la gerencia va a aplicar.

**La entrega debe contener:**

- Los umbrales que separan los tramos los define su equipo, pero deben aparecer declarados como valores explícitos y localizables, no incrustados dentro de la lógica.
- La clasificación debe emplear al menos dos indicadores distintos; un solo indicador no sustenta una política de crédito.
- El resultado debe mostrar la clasificación por empresa y el recuento de clientes en cada tramo.

**Criterio de aceptación.** Cada umbral debe poder justificarse ante un cliente que reclame su clasificación. Umbrales elegidos sin criterio explicable se califican como no logrados aunque el código funcione.

In [14]:
# =================================================
# EJERCICIO 5: POLÍTICA DE CRÉDITO DE ANDES SUPPLY
# =================================================

print("\n" + "=" * 48)
print(" Clasificación de clientes por tramo de crédito")
print("=" * 48, "\n")

# Clasificar a las 14 mineras en 3 tramos comerciales basándonos
# en su liquidez (capacidad de pago a corto plazo) y solvencia (riesgo de deuda).

# 1. Declaración explícita de umbrales
# Definimos reglas claras y justificables ante un cliente que reclame.
# Un cliente sólido debe tener buena liquidez (RC >= 1.2) y baja dependencia de deuda (End <= 0.50).
UMBRAL_LIQUIDEZ_ALTA = 1.20
UMBRAL_LIQUIDEZ_MINIMA = 0.80
UMBRAL_ENDEUDAMIENTO_BAJO = 0.50
UMBRAL_ENDEUDAMIENTO_ALTO = 0.70

# 2. Clasificación usando al menos dos indicadores simultáneamente.
# Un solo indicador es engañoso
# Cruzar liquidez y endeudamiento nos da una visión integral del riesgo.
politica_credito = resultado_ejercicio_3.with_columns(
    pl.when(
        # Tramo 1: Cliente sólido. Buena liquidez y baja deuda.
        (pl.col("Razon_Corriente") >= UMBRAL_LIQUIDEZ_ALTA) &
        (pl.col("Endeudamiento") <= UMBRAL_ENDEUDAMIENTO_BAJO)
    ).then(pl.lit("Crédito 90 días"))
    .when(
        # Tramo 2: Cliente regular. Liquidez aceptable o deuda moderada.
        (pl.col("Razon_Corriente") >= UMBRAL_LIQUIDEZ_MINIMA) &
        (pl.col("Endeudamiento") <= UMBRAL_ENDEUDAMIENTO_ALTO)
    ).then(pl.lit("Crédito 30 días"))
    .otherwise(
        # Tramo 3: Cliente riesgoso. Mala liquidez y/o deuda excesiva.
        pl.lit("Pago Adelantado")
    ).alias("Tramo_Credito")
)

# 3. Presentación del resultado: Clasificación individual y recuento por tramo.
display(politica_credito.select(["NombreEmpresa", "Razon_Corriente", "Endeudamiento", "Tramo_Credito"]).sort("Tramo_Credito"))

print("\n" + "=" * 23)
print(" Resumen de la cartera")
print("=" * 23, "\n")

recuento_tramos = politica_credito.group_by("Tramo_Credito").len().sort("Tramo_Credito")
display(recuento_tramos)


 Clasificación de clientes por tramo de crédito



NombreEmpresa,Razon_Corriente,Endeudamiento,Tramo_Credito
str,f64,f64,str
"""COMPAÑIA MINERA PODEROSA S.A.A…",0.8132,0.2593,"""Crédito 30 días"""
"""MINERA ANDINA DE EXPLORACIONES…",2.1267,0.5201,"""Crédito 30 días"""
"""SHOUGANG HIERRO PERU S.A.A.""",2.585,0.6413,"""Crédito 30 días"""
"""VOLCAN COMPAÑIA MINERA S.A.A.""",1.5929,0.5817,"""Crédito 30 días"""
"""COMPAÑIA DE MINAS BUENAVENTURA…",1.5211,0.2331,"""Crédito 90 días"""
"""COMPAÑIA MINERA SANTA LUISA S.…",3.6666,0.3423,"""Crédito 90 días"""
"""MINSUR S.A.""",1.4813,0.3497,"""Crédito 90 días"""
"""NEXA RESOURCES PERU S.A.A.""",2.5857,0.2968,"""Crédito 90 días"""
"""SOCIEDAD MINERA CERRO VERDE S.…",3.5564,0.1557,"""Crédito 90 días"""



 Resumen de la cartera



Tramo_Credito,len
str,u32
"""Crédito 30 días""",4
"""Crédito 90 días""",8
"""Pago Adelantado""",2


**Interpretación**

Para establecer la política de crédito se utilizaron dos indicadores financieros, la Razón Corriente, como medida de capacidad para cubrir obligaciones de corto plazo, y el Endeudamiento, como medida de la participación de los pasivos en el financiamiento de los activos, también se definieron umbrales explícitos para que la clasificación pueda ser aplicada de manera consistente y sustentada ante los clientes.

Se consideró como condición favorable una Razón Corriente igual o superior a 1.20 y un Endeudamiento igual o inferior a 0.50. Bajo estos criterios, una empresa presenta una posición de liquidez razonable y una menor dependencia del financiamiento mediante pasivos, por lo que puede acceder al plazo máximo de 90 días, para el tramo intermedio se estableció una Razón Corriente mínima de 0.80 y un Endeudamiento máximo de 0.70. Las empresas que cumplen estas condiciones, pero no alcanzan simultáneamente los criterios del tramo superior, reciben crédito a 30 días.

Finalmente, las empresas que no cumplen las condiciones mínimas establecidas son clasificadas como Pago Adelantado, debido a que presentan una combinación financiera que, bajo la política definida, representa un mayor nivel de riesgo para otorgar crédito.

Al aplicar estas reglas sobre las 14 empresas mineras evaluables, 8 fueron clasificadas con crédito a 90 días, 4 con crédito a 30 días y 2 con pago adelantado, entre los casos que requieren mayor atención se encuentra NEXA RESOURCES ATACOCHA S.A.A., cuyo Endeudamiento alcanza 92.15 %, por encima del límite establecido de 70 %, mientras que PERUBAR S.A. presenta una Razón Corriente de 0.741, inferior al mínimo de 0.80. Estos resultados muestran cómo el uso conjunto de ambos indicadores permite establecer condiciones de crédito diferenciadas en lugar de otorgar el mismo plazo a todos los clientes.


## Ejercicio 6 — Consulta reproducible de la cartera en SQL (2 puntos)

**Resultado exigido.** La cartera resultante debe poder consultarse por personas que leen SQL y no Python. Formule sobre el perfil una consulta que devuelva las empresas con razón corriente igual o mayor que uno y endeudamiento inferior a 0,60, ordenadas de mayor a menor rentabilidad sobre patrimonio.

**La entrega debe contener:**

- Al menos cuatro columnas en la selección.
- Dos condiciones de filtrado enlazadas entre sí.
- Una columna calculada que etiquete el nivel de riesgo según las condiciones que su equipo defina.
- Ordenamiento explícito del resultado.

**Criterio de aceptación.** El resultado debe contrastarse con la clasificación construida en el ejercicio anterior. La coincidencia o discrepancia entre ambos caminos es lo que se interpreta en la pregunta escrita 6.1.

In [15]:
# ==============================================================================
# EJERCICIO 6: CONSULTA REPRODUCIBLE DE LA CARTERA EN SQL
# ==============================================================================

print("\n" + "=" * 44)
print(" Consulta SQL: cartera de clientes elegibes")
print("=" * 44, "\n")

# Permitir que analistas que leen SQL (pero no Python) puedan consultar
# la cartera de clientes mineros bajo criterios de negocio específicos.

import duckdb

# 1. Registramos el dataframe de Polars en DuckDB para poder consultarlo vía SQL.
# DuckDB permite ejecutar SQL directamente sobre dataframes de Python
# sin necesidad de exportarlos a una base de datos externa. Esto mantiene el análisis
# reproducible y accesible para usuarios que dominan SQL pero no Python.
con = duckdb.connect()
con.register("perfil_sql", resultado_ejercicio_3)

# 2. Construimos la consulta SQL solicitada.
# - SELECT: Traemos al menos 4 columnas (NombreEmpresa, Moneda, Razon_Corriente, Endeudamiento, ROE).
# - WHERE: Filtramos por liquidez mínima (RC >= 1) y endeudamiento controlado (< 0.60).
# - CASE WHEN: Etiquetamos el nivel de riesgo según umbrales que definimos.
# - ORDER BY: Ordenamos por mayor ROE para priorizar a los más rentables dentro del rango seguro.
query_sql = """
    SELECT
        NombreEmpresa,
        Moneda,
        Razon_Corriente,
        Endeudamiento,
        ROE,
        CASE
            WHEN Razon_Corriente >= 1.5 AND Endeudamiento < 0.40 THEN 'Riesgo Muy Bajo'
            WHEN Razon_Corriente >= 1.2 AND Endeudamiento < 0.50 THEN 'Riesgo Bajo'
            WHEN Razon_Corriente >= 1.0 AND Endeudamiento < 0.60 THEN 'Riesgo Moderado'
            ELSE 'Riesgo Alto'
        END AS Nivel_Riesgo
    FROM perfil_sql
    WHERE Razon_Corriente >= 1.0 AND Endeudamiento < 0.60
    ORDER BY ROE DESC
"""

# 3. Ejecutamos la consulta y convertimos el resultado a un dataframe de Polars.
resultado_sql = con.execute(query_sql).pl()

print(f"Empresas que cumplen los criterios de liquidez y endeudamiento: {resultado_sql.height}")
display(resultado_sql)

# 4. Contraste con la clasificación (Polars).
# Verificamos cuántas empresas del SQL aparecen en cada tramo del Ejercicio 5,
# para entender si los criterios son coherentes o si hay discrepancias que deban explicarse.

print("\n" + "=" * 32)
print(" Contraste con la clasificación")
print("=" * 32, "\n")

empresas_sql = resultado_sql["NombreEmpresa"].to_list()
empresas_politica = politica_credito.filter(pl.col("NombreEmpresa").is_in(empresas_sql))

print(f"Empresas del SQL que también están en la política de crédito: {empresas_politica.height}")
display(
    empresas_politica.select(["NombreEmpresa", "Tramo_Credito", "Razon_Corriente", "Endeudamiento", "ROE"])
    .sort("Tramo_Credito")
)

# 5. Identificamos empresas que están en el SQL pero NO en los tramos "90 días" o "30 días".
# Esto revela si hay discrepancias entre los criterios de ambos ejercicios.
empresas_en_sql = set(resultado_sql["NombreEmpresa"].to_list())
empresas_en_90_30 = set(
    politica_credito.filter(pl.col("Tramo_Credito").is_in(["Crédito 90 días", "Crédito 30 días"]))["NombreEmpresa"].to_list()
)

discrepancias = empresas_en_sql - empresas_en_90_30
if discrepancias:
    print(f"\n⚠️ Discrepancia detectada: {len(discrepancias)} empresa(s) aparecen en el SQL pero no en los tramos '90 días' o '30 días':")
    print(discrepancias)
else:
    print("\n✅ Coherencia total: Todas las empresas del SQL están clasificadas en 'Crédito 90 días' o 'Crédito 30 días' del ejercicio anterior")


 Consulta SQL: cartera de clientes elegibes

Empresas que cumplen los criterios de liquidez y endeudamiento: 10


NombreEmpresa,Moneda,Razon_Corriente,Endeudamiento,ROE,Nivel_Riesgo
str,str,f64,f64,f64,str
"""COMPAÑIA MINERA SANTA LUISA S.…","""Soles""",3.6666,0.3423,0.2984,"""Riesgo Muy Bajo"""
"""MINSUR S.A.""","""D lares""",1.4813,0.3497,0.2851,"""Riesgo Bajo"""
"""SOUTHERN PERU COPPER CORPORATI…","""D lares""",2.4368,0.2085,0.2279,"""Riesgo Muy Bajo"""
"""SOCIEDAD MINERA CERRO VERDE S.…","""D lares""",3.5564,0.1557,0.1405,"""Riesgo Muy Bajo"""
"""VOLCAN COMPAÑIA MINERA S.A.A.""","""D lares""",1.5929,0.5817,0.137,"""Riesgo Moderado"""
"""COMPAÑIA DE MINAS BUENAVENTURA…","""D lares""",1.5211,0.2331,0.1188,"""Riesgo Muy Bajo"""
"""SOCIEDAD MINERA EL BROCAL S.A.…","""D lares""",1.8253,0.3893,0.0849,"""Riesgo Muy Bajo"""
"""NEXA RESOURCES PERU S.A.A.""","""D lares""",2.5857,0.2968,-0.0207,"""Riesgo Muy Bajo"""
"""MINERA ANDINA DE EXPLORACIONES…","""Soles""",2.1267,0.5201,-0.0268,"""Riesgo Moderado"""



 Contraste con la clasificación

Empresas del SQL que también están en la política de crédito: 10


NombreEmpresa,Tramo_Credito,Razon_Corriente,Endeudamiento,ROE
str,str,f64,f64,f64
"""MINERA ANDINA DE EXPLORACIONES…","""Crédito 30 días""",2.1267,0.5201,-0.0268
"""VOLCAN COMPAÑIA MINERA S.A.A.""","""Crédito 30 días""",1.5929,0.5817,0.137
"""COMPAÑIA DE MINAS BUENAVENTURA…","""Crédito 90 días""",1.5211,0.2331,0.1188
"""COMPAÑIA MINERA SANTA LUISA S.…","""Crédito 90 días""",3.6666,0.3423,0.2984
"""MINSUR S.A.""","""Crédito 90 días""",1.4813,0.3497,0.2851
"""NEXA RESOURCES PERU S.A.A.""","""Crédito 90 días""",2.5857,0.2968,-0.0207
"""SOCIEDAD MINERA CERRO VERDE S.…","""Crédito 90 días""",3.5564,0.1557,0.1405
"""SOCIEDAD MINERA CORONA S.A.""","""Crédito 90 días""",1.529,0.3457,-0.0285
"""SOCIEDAD MINERA EL BROCAL S.A.…","""Crédito 90 días""",1.8253,0.3893,0.0849



✅ Coherencia total: Todas las empresas del SQL están clasificadas en 'Crédito 90 días' o 'Crédito 30 días' del ejercicio anterior


**Interpretación**

La consulta SQL permitió reproducir la selección de clientes utilizando directamente los criterios establecidos en el ejercicio, una Razón Corriente igual o superior a 1.00 y un Endeudamiento inferior a 0.60; además, mediante CASE WHEN se incorporó una clasificación del nivel de riesgo y se ordenaron los resultados de mayor a menor ROE. En total, 10 de las 14 empresas mineras cumplen simultáneamente las dos condiciones establecidas en la consulta.

Al contrastar estos resultados con la clasificación obtenida en el Ejercicio 5, se observa que las 10 empresas seleccionadas por SQL también se encuentran dentro de los grupos de Crédito 90 días o Crédito 30 días; sin embargo, los criterios no son idénticos, por lo que no se esperaba una coincidencia total;por ejemplo, MINERA ANDINA DE EXPLORACIONES S.A. presenta una Razón Corriente de 2.1267 y un Endeudamiento de 0.5201, por lo que cumple el filtro SQL y fue clasificada con Crédito 30 días en la política anterior. En cambio, COMPAÑIA MINERA PODEROSA S.A.A. recibió Crédito 30 días en el Ejercicio 5, pero no aparece en la consulta SQL porque su Razón Corriente es 0.8132, inferior al mínimo de 1.00 solicitado.

Por lo tanto, existe coherencia entre ambos caminos, pero no equivalencia absoluta, el SQL reproduce correctamente los criterios específicos solicitados para esta consulta y permite comprobar que la cartera obtenida mediante una herramienta diferente mantiene consistencia con la clasificación general realizada anteriormente.


# **FASE 6 CRISP-DM: Despliegue**

## **Preguntas escritas de interpretación**

Las preguntas escritas no otorgan puntaje separado: son la evidencia con la que se califica la interpretación dentro de cada criterio de la rúbrica. Un notebook que ejecuta correctamente pero no interpreta no alcanza el nivel Excelente en ningún criterio.

El criterio «Escalera analítica» se evalúa únicamente con la pregunta escrita 2.1, que no tiene ejercicio de código asociado.

## Pregunta 1.1 — Qué preguntas de negocio permite y no permite responder el conjunto de datos

El conjunto de datos permite analizar la situación financiera de las empresas mineras a partir de información pública reportada a la SMV, con los datos disponibles podemos comparar indicadores como rentabilidad, liquidez y endeudamiento, identificar qué empresas presentan mejores o peores resultados financieros y utilizar estos indicadores como evidencia para establecer condiciones de crédito diferenciadas.

También permite identificar empresas con señales de mayor riesgo financiero, por ejemplo, cuando presentan un nivel elevado de endeudamiento o una capacidad limitada para cubrir sus obligaciones de corto plazo; de esta manera, la información sirve como una base objetiva para apoyar la decisión sobre si otorgar 90 días, 30 días o solicitar pago adelantado.

Sin embargo, el conjunto de datos no permite conocer aspectos internos del cliente que también pueden influir en el riesgo de crédito: por ejemplo, no permite saber si una empresa tiene actualmente facturas vencidas con Andes Supply, su comportamiento histórico de pago con otros proveedores, sus flujos de caja futuros, sus contratos comerciales, sus garantías disponibles o la calidad crediticia de cada factura. Tampoco permite afirmar que una empresa vaya a pagar o incumplir una obligación en el futuro.

Por ello, los resultados deben entenderse como una evaluación financiera basada en información pública y no como una predicción definitiva del comportamiento de pago de cada cliente.

## Pregunta 2.1 — Clasificación de seis preguntas en los escalones de la escalera analítica

Las seis preguntas pueden ubicarse en diferentes niveles de la escalera analítica de acuerdo con el tipo de información que buscan obtener:

| Pregunta                                                                                    | Escalón          | Justificación                                                                                          |
| ------------------------------------------------------------------------------------------- | ---------------- | ------------------------------------------------------------------------------------------------------ |
| ¿Qué empresas mineras reportan información <br>financiera y en qué monedas?                     | **Descriptivo**  | Busca conocer qué ocurrió y cómo está compuesto el conjunto de <br>datos.                                  |
| ¿Qué empresa presenta el mayor ROE?                                                         | **Descriptivo**  | Identifica una característica observable en los datos históricos.                                      |
| ¿Por qué una empresa tiene un ROE elevado <br>pero puede presentar mayor riesgo financiero?     | **Diagnóstico**  | Busca explicar una situación observada relacionando el ROE con <br>endeudamiento y apalancamiento.         |
| ¿Qué condición de crédito corresponde a cada<br> empresa según sus indicadores?                 | **Prescriptivo** | Utiliza los resultados del análisis para recomendar una acción <br>concreta de negocio.                    |
| ¿Qué podría ocurrir con el resultado de una<br> empresa si cambia la cotización de los metales? | **Predictivo**   | Busca estimar un posible comportamiento futuro a partir de una <br>variable relacionada con el entorno.    |
| ¿Qué empresa pagará efectivamente sus facturas a tiempo?                                    | **Predictivo**   | Requeriría información histórica de pagos y otras variables que no <br>están disponibles en este conjunto. |


El laboratorio permite avanzar desde el análisis descriptivo hasta una decisión prescriptiva de crédito; sin embargo, no permite desarrollar un modelo predictivo confiable del incumplimiento porque no contamos con variables históricas de pago, morosidad, flujo de caja u otras características específicas de los clientes; por esta razón, no sería correcto presentar la clasificación realizada como una predicción de incumplimiento.

## Pregunta 3.1 — Qué comparaciones invalida la convivencia de dos monedas en el sector

La presencia de empresas que reportan en Soles y otras en Dólares impide realizar comparaciones directas de magnitudes monetarias absolutas sin realizar previamente una conversión a una moneda común.

Por ejemplo, no sería correcto ordenar las empresas simplemente por Activo Total, Patrimonio o Pasivo Total y concluir que una empresa es financieramente más grande que otra, porque una parte de los valores está expresada en Soles y otra en Dólares, tampoco sería válido sumar directamente los activos o ingresos de todas las empresas del sector mientras se mantengan las dos monedas mezcladas.

En cambio, sí es posible comparar indicadores financieros expresados como razones, como el ROE, ROA, Margen Neto, Razón Corriente y Endeudamiento, porque en estas operaciones los valores monetarios se dividen entre otros valores de la misma empresa y la unidad monetaria se cancela.

Por lo tanto, la convivencia de las dos monedas no impide todo el análisis financiero, pero sí limita las comparaciones de montos absolutos mientras no se realice una homologación monetaria.

## Pregunta 4.1 — Por qué el ROE más alto del sector no identifica al cliente más sólido

El mayor ROE del conjunto corresponde a NEXA RESOURCES ATACOCHA S.A.A., con un ROE de aproximadamente 119.8 %. A primera vista, este resultado podría interpretarse como una rentabilidad excepcional para los accionistas. Sin embargo, el ROE no debe analizarse de manera aislada.

En este caso, la empresa presenta un Endeudamiento de 92.15 % y un Apalancamiento de 11.75 veces, mientras que su Patrimonio Total es de aproximadamente 8,459 frente a un Pasivo Total de 99,355, esto muestra que existe una participación muy elevada de los pasivos en comparación con el patrimonio.

Por ello, un ROE elevado no significa necesariamente que la empresa sea la más sólida para otorgarle crédito, una rentabilidad alta puede estar acompañada de una estructura financiera más riesgosa. En este caso, el análisis conjunto de rentabilidad y estructura financiera permite identificar que el elevado ROE debe interpretarse con cautela antes de tomar una decisión comercial.

Desde una perspectiva gerencial, la empresa puede ser rentable para sus accionistas y, al mismo tiempo, representar un mayor nivel de riesgo para un proveedor que le otorga crédito.

## Pregunta 5.1 — Relación entre la cotización de metales y el resultado de dos empresas, sin afirmar causalidad

Durante el periodo analizado, las cotizaciones internacionales de los principales metales mostraron una evolución positiva. Según los datos obtenidos del BCRP, entre enero de 2023 y diciembre de 2024 el precio del estaño aumentó aproximadamente 23.1 %, mientras que el cobre aumentó 5.0 %. Este comportamiento constituye un contexto relevante para interpretar los resultados financieros de empresas mineras vinculadas a estos metales.

En el caso de MINSUR S.A., empresa con importante exposición al estaño, se observa para 2024 un margen neto de 44.64 %, un ROA de 18.54 % y un ROE de 28.51 %. La evolución positiva de la cotización del estaño es consistente con un contexto favorable para la empresa y puede contribuir a interpretar su desempeño financiero. Sin embargo, no puede afirmarse que el incremento del precio del estaño haya causado directamente estos resultados, debido a que también intervienen factores como los niveles de producción, costos operativos, volúmenes comercializados, tipo de cambio y estructura financiera.

Por su parte, Southern Peru Copper Corporation, vinculada principalmente al cobre, presentó en 2024 un margen neto de 31.59 %, un ROA de 18.04 % y un ROE de 22.79 %. Estos resultados se observan en un contexto en el que la cotización del cobre registró un incremento de 5.0 % entre enero de 2023 y diciembre de 2024. Al igual que en el caso anterior, existe una coincidencia temporal entre la evolución positiva del metal y el desempeño financiero observado, pero esta información por sí sola no permite establecer una relación causal, ya que los resultados empresariales dependen de múltiples factores.

En conclusión, la evolución de las cotizaciones de los metales proporciona un contexto externo útil para interpretar el desempeño financiero, pero no es suficiente para atribuir directamente los resultados de las empresas a los cambios en los precios de los commodities.

## Pregunta 6.1 — Contraste entre la cartera obtenida con SQL y la clasificación construida con Polars

La consulta SQL identificó 10 empresas que cumplen simultáneamente con una Razón Corriente igual o superior a 1.00 y un Endeudamiento inferior a 0.60. Estas empresas también aparecen dentro de los grupos de Crédito 90 días o Crédito 30 días definidos mediante Polars en el Ejercicio 5.

Sin embargo, los resultados no son exactamente iguales porque ambos ejercicios utilizan criterios diferentes. En el Ejercicio 5 se utilizaron umbrales más específicos para definir las condiciones comerciales: Razón Corriente de 1.20 y Endeudamiento de 0.50 para el tramo de 90 días, y valores mínimos de 0.80 y máximos de 0.70 para el tramo de 30 días. En cambio, la consulta SQL utiliza como filtro una Razón Corriente mínima de 1.00 y un Endeudamiento inferior a 0.60.

Un ejemplo es MINERA ANDINA DE EXPLORACIONES S.A., que presenta una Razón Corriente de 2.1267 y un Endeudamiento de 0.5201. Por cumplir los criterios SQL aparece en la cartera consultada, mientras que en la política de crédito fue clasificada con 30 días. Otro caso es COMPAÑIA MINERA PODEROSA S.A.A., cuya Razón Corriente de 0.8132 le permitió obtener 30 días en la política del Ejercicio 5, pero no cumple el filtro SQL porque su Razón Corriente es menor que 1.00.

Por tanto, existe coherencia entre ambos resultados, pero no una coincidencia total. El contraste demuestra que SQL reproduce correctamente el filtro solicitado y que las diferencias encontradas se explican por los distintos umbrales utilizados, no por un error en el procesamiento de los datos.

---

# **Informe ejecutivo A a F**






## **A. Política recomendada**

Tras analizar los estados financieros individuales 2024 de 14 empresas mineras reportantes a la SMV, Andes Supply S.A.C. debe reemplazar su política comercial actual (crédito único de 90 días a todos los clientes) por una política de crédito diferenciada en tres tramos, sustentada en dos indicadores financieros: **Razón Corriente (RC)** como medida de liquidez y **Endeudamiento** como medida de solvencia.

Los umbrales declarados y su justificación son:

| Tramo | Condición | Empresas | % de la cartera |
|---|---|---|---|
| Crédito a 90 días | RC ≥ 1.20 y Endeudamiento ≤ 0.50 | 8 | 57% |
| Crédito a 30 días | RC ≥ 0.80 y Endeudamiento ≤ 0.70 | 4 | 29% |
| Pago adelantado | No cumple ninguno de los anteriores | 2 | 14% |

**Clasificación detallada por empresa:**

| Empresa | RC | Endeudamiento | Tramo |
|---|---|---|---|
| Compañía de Minas Buenaventura S.A.A. | 1.52 | 0.23 | 90 días |
| Compañía Minera Santa Luisa S.A. | 3.67 | 0.34 | 90 días |
| Minsur S.A. | 1.48 | 0.35 | 90 días |
| Nexa Resources Perú S.A.A. | 2.59 | 0.30 | 90 días |
| Sociedad Minera Cerro Verde S.A.A. | 3.56 | 0.16 | 90 días |
| Sociedad Minera Corona S.A. | 1.53 | 0.35 | 90 días |
| Sociedad Minera El Brocal S.A.A. | 1.83 | 0.39 | 90 días |
| Southern Peru Copper Corporation | 2.44 | 0.21 | 90 días |
| Compañía Minera Poderosa S.A.A. | 0.81 | 0.26 | 30 días |
| Minera Andina de Exploraciones S.A.A. | 2.13 | 0.52 | 30 días |
| Shougang Hierro Perú S.A.A. | 2.59 | 0.64 | 30 días |
| Volcán Compañía Minera S.A.A. | 1.59 | 0.58 | 30 días |
| Nexa Resources Atacocha S.A.A. | 1.33 | 0.92 | Pago adelantado |
| Perubar S.A. | 0.74 | 0.24 | Pago adelantado |

Esta política se fundamenta en la evidencia observada en el perfil financiero del sector: empresas como Southern Peru Copper Corporation (RC = 2.44, Endeudamiento = 0.21) o Compañía de Minas Buenaventura (RC = 1.52, Endeudamiento = 0.23) presentan holgada capacidad de pago y baja dependencia de pasivos, por lo que justifican el plazo máximo. En el extremo opuesto, Perubar S.A. (RC = 0.74) no cubre sus obligaciones de corto plazo con sus activos corrientes, y Nexa Resources Atacocha S.A.A. (Endeudamiento = 0.92) depende en 92% de financiamiento externo, lo que los ubica en el tramo más restrictivo.

La política propuesta es defendible ante el cliente, reproducible (los umbrales están declarados como parámetros explícitos en el código) y alinea el riesgo comercial con la evidencia financiera pública.



## **B. Explicación al cliente**

A continuación se presentan tres ejemplos de comunicación comercial, uno por cada tramo de la política, que Andes Supply utilizaría para sustentar la clasificación ante sus clientes mineros.

**Caso 1 — Southern Peru Copper Corporation (Crédito a 90 días)**

> "Estimado cliente, su empresa ha sido clasificada en el tramo de crédito a 90 días. Esta condición se sustenta en los estados financieros individuales 2024 reportados a la SMV, que muestran una Razón Corriente de 2.44 (muy por encima del umbral de 1.20 establecido) y un nivel de Endeudamiento de 0.21, lo que refleja una estructura de capital sólida y una alta capacidad para atender sus obligaciones de corto plazo. Mantendremos esta condición mientras su situación financiera se preserve en estos rangos."

**Caso 2 — Compañía Minera Poderosa S.A.A. (Crédito a 30 días)**

> "Estimado cliente, su empresa ha sido clasificada en el tramo de crédito a 30 días. El análisis de sus estados financieros 2024 muestra una Razón Corriente de 0.81 y un Endeudamiento de 0.26. Si bien su nivel de deuda es bajo, su liquidez de corto plazo se encuentra por debajo del umbral de 1.20 requerido para el tramo de 90 días, lo que nos lleva a ajustar el plazo comercial. Esta condición será revisable cuando su razón corriente mejore sostenidamente."

**Caso 3 — Nexa Resources Atacocha S.A.A. (Pago Adelantado)**

> "Estimado cliente, la clasificación de su cuenta corresponde a pago adelantado. El análisis de sus estados financieros 2024 reporta un nivel de Endeudamiento de 92.15% (muy por encima del umbral de 70% establecido) y un apalancamiento de 11.75 veces, lo que indica una elevada dependencia de financiamiento externo respecto a su patrimonio. Aunque su rentabilidad sobre patrimonio (ROE) es alta (119.79%), este indicador está influido por el alto apalancamiento y no refleja por sí solo solidez financiera. Las condiciones comerciales aplicables son de pago adelantado, revisables cuando su estructura de capital se normalice."



## **C. Estadio de madurez de Andes Supply**

De acuerdo con los cinco estadios de evolución analítica de Davenport, Andes Supply se encuentra actualmente en el **Estadio 2: Cultivo analítico**, con señales incipientes de transición al Estadio 3.

**Evidencia que sustenta esta clasificación:**

- **Comportamiento previo (pre-analítico):** hasta 2025, la empresa otorgaba crédito a 90 días a todos los clientes sin diferenciación, decisión basada en costumbre comercial y no en evidencia. Esta práctica derivó en impagos que motivaron la revisión actual.
- **Señales de cultivo analítico (estadio actual):** la gerencia ha decidido usar datos públicos (SMV) e indicadores financieros objetivos (RC y Endeudamiento) para segmentar clientes. Esto demuestra un primer paso formal hacia una cultura basada en evidencia.

**Limitaciones que impiden alcanzar el estadio 3 (orientación analítica):**

- No existe data interna histórica consolidada de comportamiento de pagos de los clientes.
- No hay modelos predictivos de riesgo de impago.
- No se integran fuentes internas (ERP, cuentas por cobrar) con fuentes externas (SMV, BCRP).
- No hay analistas dedicados ni procesos automatizados de scoring.

El presente laboratorio representa, por tanto, el primer hito formal en la ruta de maduración analítica de Andes Supply.



## **D. Modelo DELTA Plus en siete dimensiones**

A continuación se evalúa a **Andes Supply** (no a las mineras clientes) en las siete dimensiones del modelo DELTA Plus de Davenport:

| Dimensión | Evaluación actual |
|---|---|
| **D – Data (Datos)** | Dependencia exclusiva de fuentes externas públicas (SMV). No se cuenta con data interna consolidada <br>de comportamiento de pagos, contratos o garantías. |
| **E – Enterprise (Alta dirección)** | La gerencia general impulsó el cambio tras los impagos de 2025, lo que demuestra compromiso ejecutivo.<br> Sin embargo, no hay una estrategia analítica formal documentada ni objetivos cuantificados; por ejemplo, <br>"reducir impagos en 50%". |
| **L – Leadership (Liderazgo)** | El liderazgo es reactivo (responde a problemas) más que proactivo. No hay un campeón analítico designado<br> ni sponsor ejecutivo dedicado al proyecto de maduración analítica. |
| **T – Targets (Objetivos)** | El objetivo es reducir el riesgo de impago clasificando clientes, pero no está cuantificado ni alineado <br>formalmente con KPIs financieros específicos de crecimiento o rentabilidad. |
| **A – Analysts (Analistas)** | No hay analistas dedicados. La decisión recae en la gerencia general sin equipo de crédito estructurado ni<br> formación analítica especializada. |
| **P – Processes (Procesos)** | El proceso comercial era uniforme (90 días a todos). Ahora se segmenta manualmente con base en <br>indicadores, pero sin automatización ni gobernanza formal del dato. |
| **C – Culture (Cultura)** | Transición de cultura intuitiva a cultura basada en evidencia. Probable resistencia comercial de clientes <br>grandes acostumbrados al plazo único de 90 días. |

**Diagnóstico global:** Andes Supply presenta madurez analítica baja-media, concentrada en el escalón descriptivo. Las mayores brechas están en **Data** (construir data interna), **Analysts** (formar equipo analítico) y **Targets** (definir objetivos cuantificables).



## **E. Límites del análisis**

El análisis realizado presenta los siguientes límites metodológicos que la gerencia de Andes Supply debe considerar antes de tomar decisiones definitivas:

1. **Naturaleza histórica de los datos:** los estados financieros de la SMV describen el desempeño de 2024, pero no predicen el comportamiento futuro de pago. Una empresa sólida en 2024 puede enfrentar dificultades en 2025 por caída de precios de metales, conflictos sociales u otros factores.
2. **Ausencia de data interna:** no se cuenta con información propia de Andes Supply: facturas vencidas actuales, historial real de pagos de cada cliente, contratos vigentes, garantías disponibles ni montos de exposición por cliente.
3. **No hay causalidad demostrada:** un ROE alto no causa solvencia, y un alto endeudamiento no garantiza incumplimiento. El análisis identifica correlaciones y señales, no relaciones causa-efecto.
4. **Rezago de la información pública:** los estados financieros anuales auditados pueden tener rezago y no capturan eventos recientes como caídas de precios de metales, paros mineros, cambios regulatorios.
5. **Cobertura parcial del sector:** el análisis solo incluye empresas que reportan a la SMV. Mineras medianas o pequeñas no reportantes quedan fuera del modelo.
6. **Moneda de reporte mixta:** las empresas reportan en soles y dólares, lo que impide comparar importes absolutos entre ellas, solo son comparables los ratios dentro de cada empresa.
7. **Escalón analítico limitado:** el análisis es descriptivo-diagnóstico (escalón 2 de la escalera analítica). No se han construido modelos predictivos de probabilidad de impago ni modelos prescriptivos de optimización de cartera.
8. **No se evaluó la madurez analítica de las mineras:** la clasificación se basa únicamente en su situación financiera, no en su capacidad operativa, tecnológica o de gobierno corporativo.

**En síntesis, este análisis NO demuestra:**

- Que una empresa vaya a pagar o incumplir en el futuro.
- Causalidad entre indicadores financieros y comportamiento de pago.
- La madurez analítica u operativa de las mineras clientes.
- Que la clasificación sea una predicción de riesgo (es una inferencia basada en evidencia pública).

La clasificación en tramos es una **inferencia razonable basada en evidencia pública**, no una predicción de comportamiento de pago.



## **F. Iniciativa priorizada**

**Iniciativa recomendada:** Construir un historial interno de comportamiento de pagos de clientes.

**Acción concreta:** implementar en el ERP de Andes Supply un módulo de tracking de cuentas por cobrar que registre, por cada cliente minero:

- Fecha de emisión de factura.
- Fecha real de pago.
- Días de atraso.
- Monto impago y motivo.
- Volumen de compras mensual.

**Justificación:** actualmente la política de crédito se basa únicamente en datos públicos (SMV). La combinación de indicadores financieros externos + historial interno de pagos permitiría:

- Migrar del estadio 2 al estadio 3 de madurez analítica (orientación analítica).
- Habilitar modelos predictivos de riesgo de impago en los próximos 12-18 meses.
- Reducir la dependencia de una sola fuente de evidencia.
- Detectar señales tempranas de deterioro en clientes actualmente clasificados en tramos favorables.

**Plazo sugerido:**

- **0-6 meses:** implementación del módulo y definición de KPIs.
- **6-12 meses:** acumulación de data histórica mínima.
- **12-18 meses:** construcción de un primer modelo predictivo de scoring de clientes.

**KPI de éxito:** reducción en al menos 50% de la tasa de impago respecto a 2025 al cierre de 2027, manteniendo el volumen de ventas.

**Riesgo a mitigar:** resistencia comercial de clientes grandes reclasificados a tramos más restrictivos. Se recomienda acompañar la implementación con un plan de comunicación comercial que explique la nueva política con base en criterios objetivos y públicos.

# **Conclusiones**

El análisis realizado permitió a Andes Supply S.A.C. transitar de una política comercial uniforme (90 días para todos los clientes) a una política de crédito diferenciada basada en evidencia financiera pública verificable.

**Hallazgos principales:**

1. **Política de crédito defendible**: La clasificación en tres tramos (90 días, 30 días y pago adelantado) se sustenta en dos indicadores financieros objetivos (Razón Corriente y Endeudamiento) con umbrales explícitos y justificables ante cualquier cliente que reclame.

2. **Lectura crítica de indicadores**: El análisis demostró que un ROE elevado no necesariamente indica solidez financiera. El caso de Nexa Resources Atacocha (ROE de 119.79% pero con 92.15% de endeudamiento) ilustra cómo un indicador aislado puede llevar a conclusiones erróneas.

3. **Limitaciones declaradas**: El análisis se basa exclusivamente en información pública de la SMV (estados financieros 2024), por lo que no predice el comportamiento futuro de pago ni evalúa la madurez analítica de las mineras clientes. La clasificación es una inferencia razonable, no una predicción.

4. **Madurez analítica incipiente**: Andes Supply se encuentra en el Estadio 2 de madurez analítica (Cultivo analítico), con brechas significativas en datos internos, equipo analítico dedicado y objetivos cuantificados.

5. **Iniciativa priorizada**: La construcción de un historial interno de comportamiento de pagos permitiría migrar al Estadio 3 y habilitar modelos predictivos en los próximos 12-18 meses.

**Valor generado**: El laboratorio demuestra que el dato público, cuando se analiza con rigor metodológico y se interpreta con límites claros, constituye un activo estratégico para la toma de decisiones comerciales, incluso en ausencia de información interna.

---

# **Declaración de uso de asistentes de IA**

Este laboratorio fue desarrollado con apoyo de **Qwen (asistente de IA)** en las siguientes partes:

- **Revisión y ajuste de la lógica de código** en los ejercicios 3, 5 y 6 (estructura de joins, uso de when/otherwise y construcción de la consulta SQL).
- **Redacción de las interpretaciones** en markdown después de cada bloque de código, a partir de los outputs obtenidos.
- **Construcción del informe ejecutivo A-F** (secciones A a F), integrando los resultados del análisis con los conceptos de madurez analítica y modelo DELTA Plus.

**Todo el código fue revisado, ejecutado y comprendido por el equipo**, y la documentación fue redactada con palabras propias justificando las decisiones metodológicas. Los valores numéricos presentados provienen directamente del cálculo sobre los datos descargados de la SMV.

---

### **Participación del Equipo**

| **N.°** | **Apellidos y nombres**  | **Código** | **Aporte principal en este laboratorio**                                                                                                                                                                                                                                                                      |
| ------: | ------------------------ | ---------: | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
|       1 | Alvino Carhuaricra, Enzo |     118430 | **Código:** Ejercicio 1 (Ficha de trazabilidad) y Ejercicio 2 (Delimitación del sector).<br><br>**Análisis:** Redacción de las preguntas 1.1 y 3.1.<br><br>**Informe:** Redacción de las secciones A (Política recomendada) y E (Límites del análisis).                                                       |
|       2 | Cordova Agurto, Isabel   |     118082 | **Código:** Ejercicio 3 (Perfil financiero con 6 indicadores) y Ejercicio 4 (Lectura crítica del ranking / ROE).<br><br>**Análisis:** Redacción de las preguntas 4.1 y 5.1.<br><br>**Informe:** Redacción de las secciones B (Explicación al cliente) y C (Estadio de madurez).                               |
|       3 | Palomino Acuña, Luis     |     119055 | **Código:** Ejercicio 5 (Política de crédito con umbrales) y Ejercicio 6 (Consulta SQL con DuckDB).<br><br>**Análisis:** Redacción de las preguntas 2.1 y 6.1.<br><br>**Informe:** Redacción de la sección D (Modelo DELTA Plus en 7 dimensiones).                                                            |
|       4 | Vasquez Espinoza, Jesus  |     110654 | **Código:** Revisión y estandarización de la documentación línea por línea de todo el notebook.<br><br>**Análisis:** Redacción de las conclusiones generales y la Declaración de uso de IA.<br><br>**Informe:** Redacción de la sección F (Iniciativa priorizada) y unificación/formato final del entregable. |
